<a href="https://colab.research.google.com/github/Vitor-coder-eng/Despertar_da_rede_neural/blob/main/document/Ir_Al%C3%A9m_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**FarmTech Solutions**🌱

Voltando um pouco no tempo. Antes da implementação oficial do nosso sistema na propriedade do fazendeiro Tex Willer, optamos por realizar testes controlados em nosso laboratório na FarmTech. Através desses testes, buscamos validar a eficácia do sistema de visão computacional.

###**⚠️ Atenção:** a célula de código a seguir **não foi executada** neste ambiente. Ela foi desenvolvida e testada previamente no VS Code, e está aqui **somente para fins ilustrativos** e didáticos.

In [ ]:
import cv2
import torch
import numpy as np
import pathlib
from pathlib import Path

# Corrige caminho para Windows
pathlib.PosixPath = pathlib.WindowsPath

# Carrega o modelo
path = 'best.pt'
model = torch.hub.load('ultralytics/yolov5', 'custom', path=path, force_reload=True)
model.conf = 0.6

# Captura da DroidCam - geralmente a câmera é 1
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Erro ao abrir a câmera. Verifique se o DroidCam está conectado.")
    exit()

print("Câmera conectada. Pressione 'q' para sair.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Não foi possível capturar o frame.")
        break

    # Converte a imagem para RGB e detecta
    results = model(frame)
    frame_detected = np.squeeze(results.render())  # Renderiza as detecções

    cv2.imshow('Detecção com YOLO', frame_detected)

    if cv2.waitKey(5) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## Justificativas Técnicas

Após apresentar o código utilizado, seguem algumas **justificativas técnicas** da implementação.

Ao invés de optarmos por um ESP32 CAM ou a própria câmera do computador que foi realizado os testes, decidimos usar uma abordagem diferente para realizar o reconhecimento em tempo real. O **DroidCam** se deu como forma de simular o uso de **câmeras de segurança** diretamente com um dispositivo móvel. Essa abordagem se mostrou acessível e eficiente para nossos testes, permitindo uma fácil conexão entre o celular e o computador via IP, dispensando extensas linhas de código (como no caso do ESP32 CAM), como demonstramos a seguir.

> 🛠️ **Tutorial - Como usar o celular como webcam com DroidCam**  
> 1. Instale o aplicativo **DroidCam** no celular (Android/iOS).  
> 2. Instale o cliente DroidCam no seu computador (Windows/Linux).  
> 3. Conecte ambos à **mesma rede Wi-Fi**.  
> 4. No celular, será exibido um **endereço IP**. Digite esse IP no cliente do PC.  
> 5. Clique em "Start". Pronto! A câmera do celular agora é reconhecida como webcam.

**Abaixo algumas imagens representando o tutorial:**

---
<p align="center">
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/phone.png" width="20%" />
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/pc3.png" width="76%" />
</p>

---

### Motivos para não funcionar no Colab:
- O Colab **não possui acesso à câmera local do usuário** ( também no caso da nossa abordagem que estamos ultilizando).
- A execução depende de bibliotecas como `cv2` e `torch`, que nem sempre são compatíveis com o ambiente virtual do Colab de forma nativa ( no caso do VS Code tivemos que usar versões específicas de bibliotecas e até uma versão específica do Python para ter compatibilidade. `Python` versão 3.10, `Numpy` versão  1.24.4).
- O arquivo `best.pt` (modelo treinado) precisa estar acessível via link/drive. O Colab não possui acesso a arquivos do computador.

> Para executar no Colab, seria necessário alterar o código e o local de acesso do arquivo `best.pt`, além de configurar uma webcam externa com permissões específicas.

---

### Comentários sobre o funcionamento do código

- O modelo `best.pt` foi carregado utilizando `torch.hub.load()` com base no repositório `ultralytics/yolov5`.
- Definimos um limiar de confiança de **0.6** com `model.conf = 0.6` para filtrar apenas detecções com maior certeza.
- Utilizamos a biblioteca `cv2` (OpenCV) para capturar o vídeo em tempo real via `cv2.VideoCapture(1)`.
- Cada frame é processado pelo modelo YOLO, e as detecções são desenhadas com `results.render()`.
- A imagem com as detecções é exibida em tempo real usando `cv2.imshow`.

---

O arquivo `best.pt`, citado algumas vezes acima, é o arquivo gerado pelo `YOLOv5`, que contém a arquitetura, os pesos e as classes treinadas.


---
### Monitores e as Últimas Justificativas Técnicas

**Para realizar o reconhecimento com as imagens, ultilizamos dois monitores exatamente no mesmo formato que as figuras a seguir:**
 - O **primeiro monitor** (o horizontal) executamos o código e acompanhamos o reconhecimento em uma janela pop-up.
 - O **segundo monitor** (o vertical) usamos para  exibir as imagens.

Após isso, apenas apontamos a câmera do celular para o monitor 2.



<p align="center">
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/monitor1.png" width="60%" />
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/monitor2.png" width="30%" />
</p>

---

### Análise dos Resultados

Com o modelo final pronto, passamos para a parte mais interessante: **os testes ao vivo**. Os resultados foram **satisfatórios e surpreendentes**, como você verá a seguir.

Como o código foi configurado para que o modelo só reconhecesse objetos com **confiança acima de 60%**, todos os prints apresentados mostram **altos níveis de acurácia**, assim, garantindo que apenas predições com alta certeza fossem consideradas. Isso evita generalizações excessivas e reduz a chance de falsos positivos.

Em decorrência desse fato, outro ponto importante é que o modelo não confundiu as classes em nenhum momento:
- **"corn"** foi sempre identificado corretamente como milho.  
- **"fox"** foi sempre identificado corretamente como raposa.  

**Nota: Algumas imagens serão apresentadas junto com as análises e conclusões do relatório, o restante das imagens dos testes estarão disponíveis em uma galeria no final deste relatório.**

---

### Experimento com grão de milho

Um experimento particularmente divertido e interessante foi realizado aqui nos laboratórios da **FarmTech**:  
Colocamos **um grão de milho em um prato** e apontamos a câmera do celular para ele. O modelo foi capaz de detectar o grão com **60% de precisão**, mesmo fora do contexto tradicional (milhos em espiga, em campo aberto).

Esse teste demonstra a capacidade do modelo de **generalizar bem**, saindo do mundo das fotos estáticas e super focadas e entrando em imagens mais realistas, **ao vivo**.

---
![Desempenho do modelo por classe](https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test_especial1.png)

---


Além disso, em algumas imagens que continham **tanto a espiga de milho quanto os grãos soltos**, o modelo foi capaz de detectar ambos. Isso nos leva a uma reflexão importante...

---
<p align="center">
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test8.png" width="50%" />
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test7.png" width="49%" />
</p>

---

### ⚠️ Alerta técnico sobre a base de dados

Apesar da empolgação da nossa equipe com os resultados, precisamos considerar o contexto real do problema:  
Nosso objetivo é detectar **raposas atacando milharais**, não **grãos de milho individualmente**.

O fato de o modelo estar identificando grãos soltos com tanta precisão pode indicar um **desequilíbrio ou excesso de especificidade na base de treinamento**.  
Isso evidencia a importância de uma **curadoria mais crítica dos dados**, especialmente para ajustar o modelo ao **propósito real** da aplicação.

---

### 🦊 Detecção da raposa: teste com vídeo

Agora vem um outro momento interessante nos testes: como a **FarmTech** obviamente não tem acesso a uma raposa real, improvisamos.  
Reproduzimos um **vídeo do YouTube com uma raposa em movimento** e apontamos nossa câmera para a tela.

**Resultado:** o sistema foi capaz de reconhecer a raposa em diferentes posições e movimentos, com **ótimo desempenho**, mesmo com a perda natural de qualidade da câmera. Isso mostra que a base para a classe *fox* está bem balanceada e oferece **bons resultados mesmo em situações adversas**.

[🔗 Clique aqui para assistir ao vídeo de teste](https://www.youtube.com/watch?v=9OhafhgdAMs)

---
 Com isso, concluimos nossas análises e conclusões desta entrega. Abaixo, montamos uma galeria com todas as outras fotos dos testes que não foram usadas até aqui.

# **Galeria de imagens dos testes**

---
<p align="center">
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test1.png" width="50%" />
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test2.png" width="49%" />
</p>
<p align="center">
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test3.png" width="50%" />
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test4.png" width="49%" />
</p>
<p align="center">
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test5.png" width="50%" />
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test6.png" width="49%" />
</p>
<p align="center">
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test9.png" width="50%" />
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test11.png" width="49%" />
</p>
<p align="center">
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test12.png" width="50%" />
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test13.png" width="49%" />
</p>
<p align="center">
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test14.png" width="50%" />
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test15.png" width="49%" />
</p>
<p align="center">
  <img src="https://raw.githubusercontent.com/Vitor-coder-eng/Despertar_da_rede_neural/main/assets/test16.png" width="50%" />

---